# Advanced Deep Learning - Exercise 5 - Graph Neural Networks



## Node Classification



In this first part of Exercise 5, we are going to take a look at graph convolutional networks to perform semi-supervised node classification on the popular Cora citation dataset. In Cora, each paper constitutes a node and edges represent citations. Each paper belongs to one out of 7 different categories. The node representations are derived from a compacted bag of words representation. In this semi-supervised learning setting we aim to predict node categories given only a few labels.

Only add code in the sections mareked as follows, the remaining code should stay untouched:

In [ ]:
import torch

print("PyTorch has version {}".format(torch.__version__))

# Install torch geometric
torch_version = str(torch.__version__)
scatter_src = f"https://pytorch-geometric.com/whl/torch-{torch_version}.html"
sparse_src = f"https://pytorch-geometric.com/whl/torch-{torch_version}.html"
!pip install torch-scatter -f $scatter_src
!pip install torch-sparse -f $sparse_src
!pip install torch-geometric
!pip install pytorch-lightning

In [ ]:
from collections import OrderedDict
from typing import List

import pytorch_lightning as pl
import torch
import torch.nn.functional as F
import torch_geometric
import torch_geometric.nn as geom_nn
import torch_geometric.transforms as T
from pytorch_lightning import LightningModule, Trainer
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.loggers import WandbLogger
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torch_geometric.datasets import Planetoid

import wandb

We make use of a fixed seed to ensure reproducibility:

In [ ]:
seed = 42
pl.seed_everything(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Let's load the *Cora* dataset and inspect the type of data:

In [ ]:
name_data = "Cora"
dataset = Planetoid(root="/tmp/" + name_data, name=name_data)
dataset.transform = T.NormalizeFeatures()

print(f"Number of Classes in {name_data}:", dataset.num_classes)
print(f"Number of Node Features in {name_data}:", dataset.num_node_features)

data = dataset[0]
data

We extract the node feature matrix `X` of dimensionality `N` x `D`.
Here `N` is the number of papers while `D` is the feature dimensionality of each paper.

We represent the adjacency using a sparse torch tensor. You do not need to operate on its dense equivalent.

In [ ]:
X = data.x
N, D = X.shape

A = torch.sparse_coo_tensor(
    data.edge_index,
    torch.ones_like(data.edge_index[0]).float(),
    (N, N),
).coalesce()

y = data.y
num_classes = torch.max(y) + 1

**Graph Convolutional Network**

We are going to implement a multi-layer graph convolutional networ (GCN) in two steps.

$$\mathbf{H}^{(l+1)} = \sigma\left(\mathbf{D}^{-\frac{1}{2}}\mathbf{A}\mathbf{D}^{-\frac{1}{2}}\mathbf{H}^{(l)}\mathbf{W}^{(l)}\right)$$

As $\sigma$ we use the ReLU activation function, except for the last layer, which outputs raw logits.


The first step is implementing a general graph convolution module in the following. Please implement its forward function while making use of the node feature matrix $\mathbf{X}$ and the normalized Laplacian $\hat{\mathbf{A}}= \mathbf{D}^{−\frac{1}{2}} \mathbf{A}\mathbf{D}^{−\frac{1}{2}}$. In the following, please implement:

$$\mathbf{Z}^{(l+1)}=\hat{\mathbf{A}} \mathbf{H}^{(l)} \mathbf{W}^{(l)}$$

In [ ]:
class GraphConvolution(nn.Module):
    """
    Graph Convolution Layer as proposed in Kipf & Welling (2017).

    Parameters
    ----------
    in_channels : int
        Dimensionality of input features.
    out_channels : int
        Dimensionality of output features.
    """

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.linear = nn.Linear(in_channels, out_channels, bias=False)

    def forward(self, x: torch.Tensor, a_hat: torch.sparse.FloatTensor) -> torch.Tensor:
        """
        Applies a linear transformation followed by propagation using a sparse normalized adjacency matrix.

        Parameters
        ----------
        x : torch.Tensor
            Node feature matrix of shape [num_nodes, in_channels].
        a_hat : torch.sparse.FloatTensor
            Normalized adjacency matrix with self-loops.

        Returns
        -------
        torch.Tensor
            Output feature matrix of shape [num_nodes, out_channels].
        """
        x = self.linear(x)
        return torch.spmm(a_hat, x)

We now have access to the general graph convolution operator. In the following we implement a multi-layer GCN. Your tasks are the following:


*   Implement the `_normalize()` function that computes $\hat{\mathbf{A}}$
*   Implement the `self.propagate` instance variable using `nn.ModuleList()`. Each hidden layer of the GCN should contain a graph convolution layer, a non-linearity and a dropout layer.
*   Implement the `forward()` method based to perform message passing according to the architecture implemented in `self.propagate`.



In [ ]:
class GCN(nn.Module):
    """
    Graph Convolution Network: as proposed in [Kipf et al. 2017](https://arxiv.org/abs/1609.02907).

    Parameters
    ----------
    n_features: int
        Dimensionality of input features.
    n_classes: int
        Number of classes for the semi-supervised node classification.
    hidden_dimensions: List[int]
        Internal number of features. `len(hidden_dimensions)` defines the number of hidden representations.
    activation: nn.Module
        The activation for each layer but the last.
    dropout: float
        The dropout probability.
    """

    def __init__(
        self,
        n_features: int,
        n_classes: int,
        hidden_dimensions: List[int] = [64],
        activation: nn.Module = nn.ReLU(),
        dropout: float = 0.5,
    ):
        super().__init__()
        self.n_features = n_features
        self.n_classes = n_classes
        self.hidden_dimensions = hidden_dimensions
        self.propagate = nn.ModuleList(
            ##########################################################
            # YOUR CODE HERE
            # Input and hidden layers
            [
                # TODO
            ]
            # Output and hidden layer
            + [
                # TODO
            ]
            ##########################################################
        )

    def _normalize(self, A: torch.sparse.FloatTensor) -> torch.sparse.FloatTensor:
        """
        For calculating $\hat{A} = 𝐷^{−\frac{1}{2}} 𝐴 𝐷^{−\frac{1}{2}}$.

        Parameters
        ----------
        A: torch.sparse.FloatTensor
            Sparse adjacency matrix with added self-loops.

        Returns
        -------
        A_hat: torch.sparse.FloatTensor
            Normalized message passing matrix
        """
        ##########################################################
        # YOUR CODE HERE
        A_hat = None  # TODO
        ##########################################################

        return A_hat

    def forward(self, X: torch.Tensor, A: torch.sparse.FloatTensor) -> torch.Tensor:
        """
        Forward method.

        Parameters
        ----------
        X: torch.tensor
            Feature matrix `X`
        A: torch.sparse.FloatTensor
            adjacency matrix `A` (with self-loops)

        Returns
        ---------
        X: torch.tensor
            The result of the last message passing step (i.e. the logits)
        """
        ##########################################################
        # YOUR CODE HERE
        # TODO
        ##########################################################

        return X

Create a three-layer GCN (2 hidden layers with dimension 64 each, 1 output layer for the class predictions).

In [ ]:
##########################################################
# YOUR CODE HERE
three_layer_gcn = None  # TODO
##########################################################

# Inspect the architecture
three_layer_gcn

Here we add a wraper over our custom model to use  PyTorch Lightning.

In [ ]:
class NodeLevelGNN(LightningModule):
    def __init__(self, model, lr=1e-3, weight_decay=5e-4):
        super().__init__()
        self.model = model
        self.lr = lr
        self.weight_decay = weight_decay

    def forward(self, X, A):
        return self.model(X, A)

    def training_step(self, batch, batch_idx):
        X, A, y, idx = batch
        logits = self(X, A)
        loss = F.cross_entropy(logits[idx], y[idx])
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        X, A, y, idx = batch
        logits = self(X, A)
        loss = F.cross_entropy(logits[idx], y[idx])
        preds = logits.argmax(dim=1)
        acc = (preds[idx] == y[idx]).float().mean()

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

        return {"val_loss": loss, "val_acc": acc}

    def configure_optimizers(self):
        return torch.optim.Adam(
            self.parameters(), lr=self.lr, weight_decay=self.weight_decay
        )

In [ ]:
class FullGraphDataset(Dataset):
    def __init__(self, X, A, y, idx):
        self.X = X
        self.A = A
        self.y = y
        self.idx = idx

    def __len__(self):
        return 1  # one full graph

    def __getitem__(self, i):
        return self.X, self.A, self.y, self.idx


def sparse_collate_fn(batch):
    # batch = [(X, A, y, idx)]
    return batch[0]

**Training**

Please compute the cross-entropy loss of the categorical predictions over the train and validation splits.

The optimizer is given to you. You are supposed to perform the backward step.

In [ ]:
train_dataset = FullGraphDataset(X, A, y, data.train_mask)
val_dataset = FullGraphDataset(X, A, y, data.val_mask)

train_loader = DataLoader(train_dataset, batch_size=1, collate_fn=sparse_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, collate_fn=sparse_collate_fn)

model = NodeLevelGNN(three_layer_gcn)

wandb_logger = WandbLogger(project="adl26_exercise_05", name="gcn-cora")

trainer = Trainer(
    max_epochs=400,
    callbacks=[EarlyStopping(monitor="val_loss", patience=10)],
    logger=wandb_logger,
)

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.validate(model, dataloaders=val_loader)
wandb.finish()

After evaluation, we would expect an accuracy of at least 70%.

## Graph Classification with GCNs

In this second part of the exercise, we will have a look at graph classification using MUTAG dataset of the TUDataset collection.

In [ ]:
from torch_geometric.data import Data
from torch_geometric.datasets import TUDataset

tu_dataset = TUDataset(root="/tmp/", name="MUTAG")

For simplicity, we will pre-process the dataset and extract data objects and sparse adjacency matrices. The dataset is quite small, so we can easily do that. In essence, we then just have to deal with lists.

In [ ]:
train_list = []
val_list = []

for index, data in enumerate(tu_dataset):
    X = data.x
    y = data.y

    edge_index = data.edge_index

    N = data.num_nodes
    A = torch.sparse_coo_tensor(
        data.edge_index, torch.ones_like(data.edge_index[0]).float(), (N, N)
    ).coalesce()

    idx = torch.arange(N)
    data = Data(x=X, edge_index=A, edge_attr=data.edge_attr, y=y)

    if index <= 150:
        train_list.append(data)
    else:
        val_list.append(data)


train_loader = torch_geometric.loader.DataLoader(train_list, batch_size=8, shuffle=True)
val_loader = torch_geometric.loader.DataLoader(val_list, batch_size=8)

batch = next(iter(train_loader))

We can now have a look at a batch:

In [ ]:
print("Shape of x", batch.x.shape)
print("Shape of edge_index", batch.edge_index.shape)

Now we implement our graph classification network. For this, we will reuse the `GCN` module defined above and a `head`.

- For the GCN, use `c_in` as the number of features, `c_hidden[:-1]` as the GCN's hidden channels and `c_hidden[-1]` as the GCNs output channels (`n_classes`).

- For the head, use `nn.Sequential()` with a `nn.Dropout` layer with `dp_rate_linear` as dropout probability and a final linear layer for the output (`c_out` is the number of output channels).

In [ ]:
class GraphClassificationModel(nn.Module):
    def __init__(self, c_in, c_hidden, c_out, dp_rate_linear=0.5, **kwargs):
        """GNNModel.

        Args:
            c_in: Dimension of input features
            c_hidden: Dimension of hidden features
            c_out: Dimension of output features (usually number of classes)
            dp_rate_linear: Dropout rate before the linear layer (usually much higher than inside the GNN)
            kwargs: Additional arguments for the GCN module

        """
        super().__init__()

        ##########################################################
        # YOUR CODE HERE
        self.gnn = None  # TODO
        self.head = None  # TODO
        ##########################################################

    def forward(self, x, edge_index, batch_idx):
        """Forward.

        Args:
            x: Input features per node
            edge_index: List of vertex index pairs representing the edges in the graph (PyTorch geometric notation)
            batch_idx: Index of batch element for each node

        """
        ##########################################################
        # YOUR CODE HERE
        # TODO: Implement the forward pass.
        # Use geom_nn.global_mean_pool for reducing the node features to graph features.
        ##########################################################
        return x

We now again define our lightning module.

In [ ]:
class GraphLevelGNN(pl.LightningModule):
    def __init__(self, **model_kwargs):
        super().__init__()

        # Saving hyperparameters
        self.save_hyperparameters()

        self.model = GraphClassificationModel(**model_kwargs)
        self.loss_module = nn.BCEWithLogitsLoss()

    def forward(self, data, mode="train"):
        x, edge_index, batch_idx = data.x, data.edge_index, data.batch
        x = self.model(x, edge_index, batch_idx)
        x = x.squeeze(dim=-1)

        preds = (x > 0).float()
        loss = self.loss_module(x, data.y.float())
        acc = (preds == data.y).sum().float() / preds.shape[0]
        return loss, acc

    def configure_optimizers(self):
        optimizer = optim.AdamW(
            self.parameters(), lr=0.001, weight_decay=0.0
        )  # High lr because of small dataset and small model
        return optimizer

    def training_step(self, batch, batch_idx):
        loss, acc = self.forward(batch, mode="train")
        self.log("train_loss", loss, batch_size=batch.num_graphs)
        self.log("train_acc", acc, batch_size=batch.num_graphs, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, acc = self.forward(batch, mode="val")
        self.log("val_loss", loss, batch_size=batch.num_graphs)
        self.log("val_acc", acc, batch_size=batch.num_graphs, prog_bar=True)

And train it.

In [ ]:
pl.seed_everything(42)


wandb_logger = WandbLogger(
    project="adl26_exercise_05",  # Set your own project name
    name="gcn-mutag",
)

# Create trainer
trainer = pl.Trainer(
    max_epochs=175,
    logger=wandb_logger,
    log_every_n_steps=1,
)

# Instantiate model
model = GraphLevelGNN(c_in=7, c_out=1, c_hidden=[32, 32, 32, 8], dp_rate_linear=0.2)

# Train and validate model
trainer.fit(model, train_loader, val_loader)
trainer.validate(model, dataloaders=val_loader)

wandb.finish()

Validation accuracy should be around 70%.

Feel free to play around with hyperparameters.